In [1]:
print("Hello, HIT140!")

Hello, HIT140!


In [2]:
!pip install requests beautifulsoup4 lxml

In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}

In [4]:
test_url = "https://fbref.com/en/matches/140e19d9/Austria-Jordan-June-16-2026-World-Cup"

response = requests.get(test_url, headers=headers)
print(response.status_code)

403


In [6]:
import pandas as pd
df = pd.read_csv("goalkeeper_stats.csv")
df.head()

,Squad,NumPlayers,MP,Starts,Min,Nineties,GA,GA90,SoTA,Saves,...,W,D,L,CS,CSPct,PKatt,PKA,PKsv,PKm,PKSavePct
0,Algeria,2,4,4,360,4.0,9,2.25,17,8,...,1,1,2,0,0.0,0,0,0,0,NaN
1,Argentina,1,8,8,810,9.0,8,0.89,28,20,...,7,0,1,2,25.0,0,0,0,0,NaN
2,Australia,2,4,4,390,4.3,3,0.69,17,14,...,1,1,2,2,50.0,0,0,0,0,NaN
3,Austria,1,4,4,360,4.0,9,2.25,22,13,...,1,1,2,0,0.0,1,0,0,1,NaN
4,Belgium,2,6,6,570,6.3,7,1.11,23,16,...,3,2,1,1,16.7,0,0,0,0,NaN


In [7]:
df['saves_per_match'] = df['Saves'] / df['MP']
df[['Squad', 'MP', 'Saves', 'saves_per_match']].head(10)

,Squad,MP,Saves,saves_per_match
0,Algeria,4,8,2.000000
1,Argentina,8,20,2.500000
2,Australia,4,14,3.500000
3,Austria,4,13,3.250000
4,Belgium,6,16,2.666667
5,Bosnia–Herz,4,6,1.500000
6,Brazil,5,14,2.800000
7,Cabo Verde,4,18,4.500000
8,Canada,5,6,1.200000
9,Colombia,5,6,1.200000


In [8]:
knockout_teams = [
    "Mexico", "South Africa", "Switzerland", "Canada", "Brazil", "Morocco",
    "United States", "Australia", "Germany", "Côte d'Ivoire", "Netherlands", "Japan",
    "Belgium", "Egypt", "Spain", "Cabo Verde", "France", "Norway",
    "Argentina", "Austria", "Colombia", "Portugal", "England", "Croatia",
    "Bosnia–Herz", "Paraguay", "Ecuador", "Sweden", "Senegal", "Algeria",
    "Congo DR", "Ghana"
]

df['stage'] = df['Squad'].apply(lambda x: 'knockout' if x in knockout_teams else 'group_stage')
df[['Squad', 'stage']].head(15)

,Squad,stage
0,Algeria,knockout
1,Argentina,knockout
2,Australia,knockout
3,Austria,knockout
4,Belgium,knockout
5,Bosnia–Herz,knockout
6,Brazil,knockout
7,Cabo Verde,knockout
8,Canada,knockout
9,Colombia,knockout


In [9]:
df['stage'].value_counts()

stage
knockout       31
group_stage    17
Name: count, dtype: int64

In [10]:
knockout_teams = [
    "Mexico", "South Africa", "Switzerland", "Canada", "Brazil", "Morocco",
    "USA", "Australia", "Germany", "Côte d'Ivoire", "Netherlands", "Japan",
    "Belgium", "Egypt", "Spain", "Cabo Verde", "France", "Norway",
    "Argentina", "Austria", "Colombia", "Portugal", "England", "Croatia",
    "Bosnia–Herz", "Paraguay", "Ecuador", "Sweden", "Senegal", "Algeria",
    "Congo DR", "Ghana"
]

df['stage'] = df['Squad'].apply(lambda x: 'knockout' if x in knockout_teams else 'group_stage')
df['stage'].value_counts()

stage
knockout       32
group_stage    16
Name: count, dtype: int64

In [11]:
sample_df = df.sample(n=30, random_state=42)
sample_df[['Squad', 'MP', 'Saves', 'saves_per_match', 'stage']]

,Squad,MP,Saves,saves_per_match,stage
27,Mexico,5,8,1.600000,knockout
40,Spain,8,9,1.125000,knockout
26,Korea Republic,3,9,3.000000,group_stage
43,Tunisia,3,6,2.000000,group_stage
24,Japan,4,12,3.000000,knockout
37,Scotland,3,8,2.666667,group_stage
12,Croatia,4,10,2.500000,knockout
19,Germany,4,5,1.250000,knockout
4,Belgium,6,16,2.666667,knockout
25,Jordan,3,8,2.666667,group_stage


In [12]:
sample_df.groupby('stage')['saves_per_match'].describe()

,count,mean,std,min,25%,50%,75%,max
stage,,,,,,,,
group_stage,8.0,3.041667,1.617734,1.333333,2.25,2.666667,3.166667,6.666667
knockout,22.0,2.435227,0.970593,1.125000,1.60,2.450000,2.950000,4.600000


In [13]:
sample_df['saves_per_match'].describe()

count    30.000000
mean      2.596944
std       1.178252
min       1.125000
25%       1.700000
50%       2.500000
75%       3.000000
max       6.666667
Name: saves_per_match, dtype: float64

In [14]:
from scipy import stats
import numpy as np

n = len(sample_df)
mean = sample_df['saves_per_match'].mean()
std_err = stats.sem(sample_df['saves_per_match'])  # standard error of the mean

ci = stats.t.interval(confidence=0.95, df=n-1, loc=mean, scale=std_err)
print(f"Sample mean: {mean:.3f}")
print(f"95% Confidence Interval: ({ci[0]:.3f}, {ci[1]:.3f})")

Sample mean: 2.597
95% Confidence Interval: (2.157, 3.037)


In [16]:
group_saves = sample_df[sample_df['stage'] == 'group_stage']['saves_per_match']
knockout_saves = sample_df[sample_df['stage'] == 'knockout']['saves_per_match']

t_stat, p_value = stats.ttest_ind(group_saves, knockout_saves, equal_var=False)

print(f"Group-stage mean: {group_saves.mean():.3f}")
print(f"Knockout mean: {knockout_saves.mean():.3f}")
print(f"t-statistic: {t_stat:.3f}")
print(f"p-value: {p_value:.4f}")

alpha = 0.05
if p_value < alpha:
    print("Reject H0: significant difference in saves per match between the two groups.")
else:
    print("Fail to reject H0: no significant difference detected.")

Group-stage mean: 3.042
Knockout mean: 2.435
t-statistic: 0.997
p-value: 0.3451
Fail to reject H0: no significant difference detected.
